In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import pandas as pd
import os
from dotenv import load_dotenv


load_dotenv()



# Configure optimized browser properties
chrome_options = webdriver.ChromeOptions()
chrome_options.page_load_strategy = 'eager'  # Speeds up interaction by omitting heavy media loading
driver = webdriver.Chrome(options=chrome_options)

search_url = os.getenv("PORTAL")
driver.get(search_url)
wait = WebDriverWait(driver, 10)

# -------------------------------------------------------------
# 1. Load the Proposal Numbers from the Target Excel File
# -------------------------------------------------------------
excel_path = os.getenv("BRONZE") + r"\RAW_MERGED - IND2.xlsx"

if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the target Excel file at: {excel_path}")

df_leads = pd.read_excel(excel_path)

if 'Proposal No.' not in df_leads.columns:
    raise KeyError("The Excel file must contain a column named 'Proposal No.'")

# Initialize tracking columns cleanly
for col in ["proposal details", "Proposal URL", "Project Details XML"]:
    if col not in df_leads.columns:
        df_leads[col] = "N/A"
    else:
        df_leads[col] = df_leads[col].astype(object).fillna("N/A")

print(f"Loaded {len(df_leads)} rows from Excel sheet. Starting optimized search loop...")
main_window = driver.current_window_handle

# -------------------------------------------------------------
# 2. Search Loop for each Proposal Number
# -------------------------------------------------------------
for idx, row in df_leads.iterrows():
    proposal_no = str(row['Proposal No.']).strip()
    
    if pd.isna(row['Proposal No.']) or proposal_no == "" or proposal_no.lower() == "nan":
        continue
        
    print(f"\n[{idx + 1}/{len(df_leads)}] Processing Proposal: {proposal_no}")
    
    try:
        if "trackYourProposal" not in driver.current_url or "proposal-details" in driver.current_url:
            driver.get(search_url)
            
        proposal_input = wait.until(
            EC.visibility_of_element_located((By.XPATH, "//input[@formcontrolname='proposalNumber']"))
        )
        proposal_input.clear()
        proposal_input.send_keys(proposal_no)
        
        search_button = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and contains(.,'Search')]"))
        )
        driver.execute_script("arguments[0].click();", search_button)
        
        try:
            WebDriverWait(driver, 7).until(
                EC.text_to_be_present_in_element((By.XPATH, "//table[@id='excel-table']/tbody/tr[1]/td[2]"), proposal_no)
            )
        except TimeoutException:
            print(f"  ⚠️ No matching records found or table timed out updating for: {proposal_no}. Skipping...")
            continue
            
        proposal_link = driver.find_element(By.XPATH, "//table[@id='excel-table']/tbody/tr[1]/td[2]/a")
        
        current_handles_count = len(driver.window_handles)
        driver.execute_script("arguments[0].click();", proposal_link)
        
        try:
            wait.until(lambda d: len(d.window_handles) > current_handles_count or "proposal-details" in d.current_url)
        except TimeoutException:
            pass
        
        details_url = "N/A"
        view_proposal_url = "N/A"
        project_details_xml = "N/A"
        
        opened_in_new_tab = len(driver.window_handles) > 1
        if opened_in_new_tab:
            details_window = [w for w in driver.window_handles if w != main_window][0]
            driver.switch_to.window(details_window)
            
        details_url = driver.current_url
        print(f"  -> Captured Details URL: {details_url}")
        
        # -------------------------------------------------------------
        # 3. Inside Details Page: Open & Wait for "View Proposal" Content
        # -------------------------------------------------------------
        try:
            # FIX: Wait explicitly for absolute layout visibility to ensure the button is fully active in DOM
            view_proposal_element = wait.until(
                EC.visibility_of_element_located((By.XPATH, "//a[contains(@class, 'btn') and contains(text(), 'View Proposal')]"))
            )
            
            # Brief structural pause for dynamic overlay spinners to clear out
            time.sleep(1.5)
            
            pre_click_windows = driver.window_handles
            
            # FIX: Reverted to JavaScript click to bypass "element not interactable" blockades gracefully
            driver.execute_script("arguments[0].click();", view_proposal_element)
            
            # Give Angular a moment to kickstart routing and structural XML creation
            print("  -> Waiting for Project Details container to settle...")
            time.sleep(3.5)
            
            try:
                container_element = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.XPATH, "//*[@id='caf_project'] | //*[contains(@id,'caf_project')]"))
                )
                project_details_xml = container_element.get_attribute('outerHTML')
                print("  -> Captured Project Details Container XML successfully.")
            except TimeoutException:
                try:
                    table_element = driver.find_element(By.XPATH, "//h4[contains(text(),'Project Details')]/following::table[1] | //table[1]")
                    project_details_xml = table_element.get_attribute('outerHTML')
                    print("  -> Captured Project Details Table XML via structure layout fallback.")
                except Exception:
                    print("  ⚠️ Could not isolate the project details layout tree structure.")

            # Track finalized URL endpoint location after loading completion
            post_click_windows = driver.window_handles
            if len(post_click_windows) > len(pre_click_windows):
                proposal_doc_window = [w for w in post_click_windows if w not in pre_click_windows][0]
                driver.switch_to.window(proposal_doc_window)
                view_proposal_url = driver.current_url
                driver.close()
                if opened_in_new_tab:
                    driver.switch_to.window(details_window)
                else:
                    driver.switch_to.window(main_window)
            else:
                view_proposal_url = driver.current_url
                
            print(f"  -> Captured View Proposal URL: {view_proposal_url}")
                    
        except (NoSuchElementException, TimeoutException) as err:
            print(f"  ⚠️ 'View Proposal' link interaction sequence encountered an exception: {err}")
            
        # -------------------------------------------------------------
        # 4. Clean Master Reset to Search Index Screen
        # -------------------------------------------------------------
        if opened_in_new_tab:
            driver.close()
            driver.switch_to.window(main_window)
            
        driver.get(search_url)
        wait.until(EC.visibility_of_element_located((By.XPATH, "//input[@formcontrolname='proposalNumber']")))
            
        df_leads.at[idx, "proposal details"] = details_url
        df_leads.at[idx, "Proposal URL"] = view_proposal_url
        df_leads.at[idx, "Project Details XML"] = project_details_xml
        
    except Exception as e:
        print(f" ❌ Global processing fail for proposal {proposal_no}: {e}")
        all_windows = driver.window_handles
        if len(all_windows) > 1:
            for extra_w in all_windows[1:]:
                driver.switch_to.window(extra_w)
                driver.close()
        driver.switch_to.window(main_window)
        driver.get(search_url)
        continue

# -------------------------------------------------------------
# 5. Post-Loop Sheet Save Back Execution
# -------------------------------------------------------------
print("\n[Final Step] Saving all collected data and URLs to the Excel file in a single batch...")
df_leads.to_excel(excel_path, index=False)
print(f"🎉 Task complete! All data successfully saved to: {excel_path}")

# Quit the WebDriver
driver.quit()

Loaded 504 rows from Excel sheet. Starting optimized search loop...

[1/504] Processing Proposal: SIA/MP/MIN/532333/2025
  -> Captured Details URL: https://parivesh.nic.in/newupgrade/#/trackYourProposal/proposal-details?proposalId=SIA%2FMP%2FMIN%2F532333%2F2025&proposal=124014170
  -> Waiting for Project Details container to settle...
  -> Captured Project Details Container XML successfully.
  -> Captured View Proposal URL: https://parivesh.nic.in/newupgrade/#/report/ec?proposalId=124014170&legacyApplicationId=0

[2/504] Processing Proposal: SIA/UP/MIN/574787/2026
  -> Captured Details URL: https://parivesh.nic.in/newupgrade/#/trackYourProposal/proposal-details?proposalId=SIA%2FUP%2FMIN%2F574787%2F2026&proposal=1223483374
  -> Waiting for Project Details container to settle...
  -> Captured Project Details Container XML successfully.
  -> Captured View Proposal URL: https://parivesh.nic.in/newupgrade/#/report/ec?proposalId=1223483374&legacyApplicationId=0

[3/504] Processing Proposal: 

In [3]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select, WebDriverWait
import re

load_dotenv()

# Configure optimized browser properties
chrome_options = webdriver.ChromeOptions()
chrome_options.page_load_strategy = (
    'eager'  # Speeds up interaction by omitting heavy media loading
)
driver = webdriver.Chrome(options=chrome_options)

search_url = os.getenv('PORTAL')
driver.get(search_url)
wait = WebDriverWait(driver, 10)

# -------------------------------------------------------------
# 1. Load the Proposal Numbers from the Target Excel File
# -------------------------------------------------------------
excel_path = os.getenv('ActiveLeads')

if not os.path.exists(excel_path):
  raise FileNotFoundError(
      f'Could not find the target Excel file at: {excel_path}'
  )

df_leads = pd.read_excel(excel_path)

if 'Proposal No.' not in df_leads.columns:
  raise KeyError("The Excel file must contain a column named 'Proposal No.'")

# Initialize tracking columns cleanly
for col in ['proposal details', 'Proposal URL', 'Project Details XML']:
  if col not in df_leads.columns:
    df_leads[col] = 'N/A'
  else:
    df_leads[col] = df_leads[col].astype(object).fillna('N/A')

print(
    f'Loaded {len(df_leads)} rows from Excel sheet. Starting optimized search'
    ' loop...'
)
main_window = driver.current_window_handle

# -------------------------------------------------------------
# 2. Search Loop for each Proposal Number
# -------------------------------------------------------------
for idx, row in df_leads.iterrows():
  proposal_no = str(row['Proposal No.']).strip()

  # Skip if Proposal No. is missing or invalid
  if (
      pd.isna(row['Proposal No.'])
      or proposal_no == ''
      or proposal_no.lower() == 'nan'
  ):
    continue

  # --- FILTER CONDITION ---
  # Check if 'proposal details' is already populated
  prop_details_val = str(row['proposal details']).strip()
  if (
      not pd.isna(row['proposal details'])
      and prop_details_val not in ['', 'N/A', 'nan', 'NaN']
  ):
    print(
        f'[{idx + 1}/{len(df_leads)}] Skipping Proposal: {proposal_no} (Already'
        ' processed)'
    )
    continue

  print(f'\n[{idx + 1}/{len(df_leads)}] Processing Proposal: {proposal_no}')

  try:
    if (
        'trackYourProposal' not in driver.current_url
        or 'proposal-details' in driver.current_url
    ):
      driver.get(search_url)

    proposal_input = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@formcontrolname='proposalNumber']")
        )
    )
    proposal_input.clear()
    proposal_input.send_keys(proposal_no)

    search_button = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[@type='submit' and contains(.,'Search')]")
        )
    )
    driver.execute_script('arguments[0].click();', search_button)

    try:
      WebDriverWait(driver, 7).until(
          EC.text_to_be_present_in_element(
              (By.XPATH, "//table[@id='excel-table']/tbody/tr[1]/td[2]"),
              proposal_no,
          )
      )
    except TimeoutException:
      print(
          '  ⚠️ No matching records found or table timed out updating for:'
          f' {proposal_no}. Skipping...'
      )
      continue

    proposal_link = driver.find_element(
        By.XPATH, "//table[@id='excel-table']/tbody/tr[1]/td[2]/a"
    )

    current_handles_count = len(driver.window_handles)
    driver.execute_script('arguments[0].click();', proposal_link)

    try:
      wait.until(
          lambda d: len(d.window_handles) > current_handles_count
          or 'proposal-details' in d.current_url
      )
    except TimeoutException:
      pass

    details_url = 'N/A'
    view_proposal_url = 'N/A'
    project_details_xml = 'N/A'

    opened_in_new_tab = len(driver.window_handles) > 1
    if opened_in_new_tab:
      details_window = [
          w for w in driver.window_handles if w != main_window
      ][0]
      driver.switch_to.window(details_window)

    details_url = driver.current_url
    print(f'  -> Captured Details URL: {details_url}')

    # -------------------------------------------------------------
    # 3. Inside Details Page: Open & Wait for "View Proposal" Content
    # -------------------------------------------------------------
    try:
      view_proposal_element = wait.until(
          EC.visibility_of_element_located((
              By.XPATH,
              "//a[contains(@class, 'btn') and contains(text(), 'View"
              " Proposal')]",
          ))
      )

      time.sleep(1.5)
      pre_click_windows = driver.window_handles

      driver.execute_script('arguments[0].click();', view_proposal_element)

      print('  -> Waiting for Project Details container to settle...')
      time.sleep(3.5)

      try:
        container_element = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((
                By.XPATH,
                "//*[@id='caf_project'] | //*[contains(@id,'caf_project')]",
            ))
        )
        project_details_xml = container_element.get_attribute('outerHTML')
        print(
            '  -> Captured Project Details Container XML successfully.'
        )
      except TimeoutException:
        try:
          table_element = driver.find_element(
              By.XPATH,
              "//h4[contains(text(),'Project Details')]/following::table[1] |"
              ' //table[1]',
          )
          project_details_xml = table_element.get_attribute('outerHTML')
          print(
              '  -> Captured Project Details Table XML via structure layout'
              ' fallback.'
          )
        except Exception:
          print(
              '  ⚠️ Could not isolate the project details layout tree'
              ' structure.'
          )

      post_click_windows = driver.window_handles
      if len(post_click_windows) > len(pre_click_windows):
        proposal_doc_window = [
            w for w in post_click_windows if w not in pre_click_windows
        ][0]
        driver.switch_to.window(proposal_doc_window)
        view_proposal_url = driver.current_url
        driver.close()
        if opened_in_new_tab:
          driver.switch_to.window(details_window)
        else:
          driver.switch_to.window(main_window)
      else:
        view_proposal_url = driver.current_url

      print(f'  -> Captured View Proposal URL: {view_proposal_url}')

    except (NoSuchElementException, TimeoutException) as err:
      print(
          "  ⚠️ 'View Proposal' link interaction sequence encountered an"
          f' exception: {err}'
      )

    # -------------------------------------------------------------
    # 4. Clean Master Reset to Search Index Screen
    # -------------------------------------------------------------
    if opened_in_new_tab:
      driver.close()
      driver.switch_to.window(main_window)

    driver.get(search_url)
    wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@formcontrolname='proposalNumber']")
        )
    )

    df_leads.at[idx, 'proposal details'] = details_url
    df_leads.at[idx, 'Proposal URL'] = view_proposal_url
    df_leads.at[idx, 'Project Details XML'] = project_details_xml

  except Exception as e:
    print(f' ❌ Global processing fail for proposal {proposal_no}: {e}')
    all_windows = driver.window_handles
    if len(all_windows) > 1:
      for extra_w in all_windows[1:]:
        driver.switch_to.window(extra_w)
        driver.close()
    driver.switch_to.window(main_window)
    driver.get(search_url)
    continue

# -------------------------------------------------------------
# 5. Post-Loop Sheet Save Back Execution
# -------------------------------------------------------------
print(
    '\n[Final Step] Saving all collected data and URLs to the Excel file in a'
    ' single batch...'
)
# Remove non-printable ASCII control characters (keeping standard newlines/tabs)
ILLEGAL_CHARACTERS_RE = re.compile(
    r"[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x9F]"
)


def clean_illegal_chars(val):
    if isinstance(val, str):
        return ILLEGAL_CHARACTERS_RE.sub("", val)
    return val


# Apply cleaning to the entire dataframe
df_leads_clean = df_leads.applymap(
    lambda x: clean_illegal_chars(x)
)  # or df_leads.map(clean_illegal_chars) in pandas >= 2.1

# Export to Excel
df_leads_clean.to_excel(excel_path, index=False)

print(f'🎉 Task complete! All data successfully saved to: {excel_path}')

driver.quit()

Loaded 3562 rows from Excel sheet. Starting optimized search loop...

[1/3562] Processing Proposal: SIA/PB/IND1/466563/2024
  -> Captured Details URL: https://parivesh.nic.in/newupgrade/#/trackYourProposal/proposal-details?proposalId=SIA%2FPB%2FIND1%2F466563%2F2024&proposal=54692644
  -> Waiting for Project Details container to settle...
  -> Captured Project Details Container XML successfully.
  -> Captured View Proposal URL: https://parivesh.nic.in/newupgrade/#/report/ec?proposalId=54692644&legacyApplicationId=0

[2/3562] Processing Proposal: SIA/PB/IND1/576135/2026
  -> Captured Details URL: https://parivesh.nic.in/newupgrade/#/trackYourProposal/proposal-details?proposalId=SIA%2FPB%2FIND1%2F576135%2F2026&proposal=1224702697
  -> Waiting for Project Details container to settle...
  -> Captured Project Details Container XML successfully.
  -> Captured View Proposal URL: https://parivesh.nic.in/newupgrade/#/report/ec-form-seven?id=1224702692&caf=1224653783&ecId=1224702697

[3/3562] Pro

IllegalCharacterError: <div _ngcontent-flh-c212="" id="caf_project" class="number-counter" style="--start-value: 0;"><h4 _ngcontent-flh-c212="" class="mb-4 font-weight-label">Project Details</h4><div _ngcontent-flh-c212="" class="card card-body bg-white"><div _ngcontent-flh-c212=""><h6 _ngcontent-flh-c212="" class="h5 mb-3"><span _ngcontent-flh-c212="" class="counter"> Details of Project</span></h6><div _ngcontent-flh-c212=""><div _ngcontent-flh-c212="" class="table-responsive number-counter-inner"><table _ngcontent-flh-c212="" class="table table-striped"><tbody _ngcontent-flh-c212=""><tr _ngcontent-flh-c212=""><td _ngcontent-flh-c212="" class="col-form-label label-font" style="width: 50%;"><div _ngcontent-flh-c212="" class="d-flex"><span _ngcontent-flh-c212="" class="counter"> Name of the Project </span></div></td><td _ngcontent-flh-c212="" class="col-form-label"> Application for Expansion (additional 4 nos. of E&amp;A wells &amp; 4 nos. of EPUs/QPUs) in "Onshore Oil &amp; Gas Explora

In [ ]:
# import re

# # Remove non-printable ASCII control characters (keeping standard newlines/tabs)
# ILLEGAL_CHARACTERS_RE = re.compile(
#     r"[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x9F]"
# )


# def clean_illegal_chars(val):
#     if isinstance(val, str):
#         return ILLEGAL_CHARACTERS_RE.sub("", val)
#     return val


# # Apply cleaning to the entire dataframe
# df_leads_clean = df_leads.applymap(
#     lambda x: clean_illegal_chars(x)
# )  # or df_leads.map(clean_illegal_chars) in pandas >= 2.1

# # Export to Excel
# df_leads_clean.to_excel(excel_path, index=False)

# print(f'🎉 Task complete! All data successfully saved to: {excel_path}')

# driver.quit()

C:\Users\Amit\AppData\Local\Temp\ipykernel_19688\3379619939.py:16: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_leads_clean = df_leads.applymap(


🎉 Task complete! All data successfully saved to: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - Silver\Temp\Non-Mineral_ActiveLeads.xlsx
